In [13]:
import json
import pandas as pd
import ast
from pathlib import Path


def extract_first(item):
    if isinstance(item, list):
        return item[0] if item else None
    return item


def safe_json_load(s):
    if isinstance(s, dict):
        return s
    try:
        return json.loads(s)
    except (json.JSONDecodeError, TypeError):
        return {}


def ensure_list(value):
    return value if isinstance(value, list) else []


def normalize_case(case_id, case):
    metadata = case.get("metadata") or case
    legal_basis = safe_json_load(extract_first(metadata.get("caseLegalBasis", [{}])))
    sectors = safe_json_load(extract_first(metadata.get("caseSectors", [{}])))
    title = extract_first(metadata.get("caseTitle"))
    companies = metadata.get("caseCompanies", [])
    companies_text = "; ".join(companies) if isinstance(companies, list) else companies
    return {
        "case_id": case_id,
        "title": title,
        "number": extract_first(metadata.get("caseNumberPart")) or extract_first(metadata.get("caseNumber")),
        "type": extract_first(metadata.get("caseType")),
        "instrument": extract_first(metadata.get("caseInstrument")),
        "dg": extract_first(metadata.get("caseDg")),
        "language": extract_first(metadata.get("language")),
        "cartel": extract_first(metadata.get("caseCartel")),
        "legal_basis_code": legal_basis.get("code") or extract_first(metadata.get("caseLegalBasisCode")),
        "legal_basis_label": legal_basis.get("label") or extract_first(metadata.get("caseLegalBasisLabel")),
        "sector_code": sectors.get("code") or extract_first(metadata.get("caseSectorsCode")),
        "sector_label": sectors.get("label") or extract_first(metadata.get("caseSectorLabel")),
        "companies": companies_text,
        "initiation_date": extract_first(metadata.get("caseInitiationDate")),
        "last_decision_date": extract_first(metadata.get("caseLastDecisionDate")),
    }


def normalize_attachments(case_id, case):
    rows = []
    for attachment in case.get("caseAttachments", []) or []:
        meta = attachment.get("metadata", {}) or {}
        rows.append({
            "case_id": case_id,
            "attachment_id": extract_first(meta.get("attachmentIdSequence")),
            "link": extract_first(meta.get("attachmentLink")),
            "language": extract_first(meta.get("language")),
            "sent_date": extract_first(meta.get("attachmentSentDate")),
            "document_date": extract_first(meta.get("attachmentDocumentDate")),
            "publication_date": extract_first(meta.get("attachmentPublicationBusinessDate")),
            "description": extract_first(meta.get("attachmentPublicationDescription")),
        })
    return rows


def normalize_decisions(case_id, case):
    rows = []
    for decision in case.get("decisions", []) or []:
        meta = decision.get("metadata") or decision
        dtype = safe_json_load(extract_first(meta.get("decisionTypes", [{}])))

        oj_publications_str = extract_first(meta.get("decisionOfficialJournalPublications"))
        oj_publications = safe_json_load(oj_publications_str).get("items", []) if oj_publications_str else []

        press_releases_str = extract_first(meta.get("decisionPressReleases"))
        press_releases = safe_json_load(press_releases_str).get("items", []) if press_releases_str else []

        press_release_pub_date = extract_first(meta.get("decisionPressReleasesPublicationDates", []))
        oj_publication_pub_date = extract_first(meta.get("decisionOfficialJournalPublicationsPublishedDates", []))

        rows.append({
            "case_id": case_id,
            "decision_number": extract_first(meta.get("decisionNumber")),
            "adoption_date": extract_first(meta.get("decisionAdoptionDate")) or extract_first(meta.get("decisionDate")),
            "type_code": dtype.get("code"),
            "type_label": dtype.get("label") or extract_first(meta.get("decisionLabel")),
            "language": extract_first(meta.get("language")),
            "oj_reference": extract_first([item.get("reference") for item in oj_publications]),
            "oj_web_description": extract_first([item.get("webDescription") for item in oj_publications]),
            "oj_published_date": oj_publication_pub_date,
            "press_release_reference": extract_first([item.get("reference") for item in press_releases]),
            "press_release_description": extract_first([item.get("webDescription") for item in press_releases]),
            "press_release_publication_date": press_release_pub_date,
        })
    return rows


def normalize_decision_attachments(case_id, case):
    rows = []
    for decision in case.get("decisions", []) or []:
        decision_id = extract_first((decision.get("metadata", {}) or {}).get("decisionNumber"))
        for attachment in decision.get("decisionAttachments", []) or []:
            meta = attachment.get("metadata", {}) or {}
            rows.append({
                "case_id": case_id,
                "decision_number": decision_id,
                "attachment_id": extract_first(meta.get("attachmentIdSequence")),
                "link": extract_first(meta.get("attachmentLink")),
                "language": extract_first(meta.get("language")),
                "sent_date": extract_first(meta.get("attachmentSentDate")),
                "document_date": extract_first(meta.get("attachmentDocumentDate")),
                "description": extract_first(meta.get("attachmentPublicationDescription")),
                "publication_date": extract_first(meta.get("attachmentPublicationBusinessDate")),
            })
    return rows


def _decision_summary(decisions):
    rows = normalize_decisions(None, {"decisions": decisions or []})
    return {
        "decision_type_labels": [row["type_label"] for row in rows if row.get("type_label")],
        "decision_adoption_dates": [row["adoption_date"] for row in rows if row.get("adoption_date")],
        "decision_publication_dates": [row["press_release_publication_date"] or row["oj_published_date"] for row in rows if row.get("press_release_publication_date") or row.get("oj_published_date")],
        "press_release_publication_dates": [row["press_release_publication_date"] for row in rows if row.get("press_release_publication_date")],
    }


def load_cases_json_df(path):
    """Load cases_merged.json as one row-per-case DataFrame with nested JSON preserved."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    raw_df = pd.DataFrame.from_dict(data, orient="index").rename_axis("case_id").reset_index()
    flat_df = pd.DataFrame([normalize_case(case_id, case) for case_id, case in data.items()])

    keep_cols = [col for col in raw_df.columns if col not in flat_df.columns or col == "case_id"]
    df = flat_df.merge(raw_df[keep_cols], on="case_id", how="left")

    for col in ["caseAttachments", "decisions"]:
        if col not in df.columns:
            df[col] = [[] for _ in range(len(df))]
        else:
            df[col] = df[col].apply(ensure_list)

    df["num_case_attachments"] = df["caseAttachments"].apply(lambda xs: len(xs or []))
    df["num_decisions"] = df["decisions"].apply(lambda xs: len(xs or []))
    df["num_decision_attachments"] = df["decisions"].apply(
        lambda decisions: sum(len(decision.get("decisionAttachments", []) or []) for decision in decisions or [])
    )

    decision_summary = pd.DataFrame(df["decisions"].apply(_decision_summary).tolist())
    return pd.concat([df, decision_summary], axis=1)


def decision_records_from_df(df):
    """Create a temporary decision-level view from df only when a decision analysis needs it."""
    rows = []
    for _, row in df.iterrows():
        for decision_row in normalize_decisions(row["case_id"], {"decisions": row.get("decisions") or []}):
            decision_row["initiation_date"] = row.get("initiation_date")
            rows.append(decision_row)
    return pd.DataFrame(rows)


In [14]:
DATA_PATH =  "cases.json"

df = load_cases_json_df(DATA_PATH)

print(f"Loaded {len(df):,} cases into one DataFrame")
print(f"Shape: {df.shape}")
df.head(1)


Loaded 10,967 cases into one DataFrame
Shape: (10967, 34)


,case_id,title,number,type,instrument,dg,language,cartel,legal_basis_code,legal_basis_label,...,caseInitiationDate,decisions,caseAttachments,num_case_attachments,num_decisions,num_decision_attachments,decision_type_labels,decision_adoption_dates,decision_publication_dates,press_release_publication_dates
0,M.2027,DEUTSCHE BANK / SAP / JV,2027,None,Merger,None,None,None,,Art. 105,...,2000-06-09,"[{'decisionDate': '2000-07-13', 'decisionLabel...",[],0,2,0,[Art. 6(1)(b)],[2000-07-13],[],[]


In [15]:
import pandas as pd
pd.set_option('display.max_columns', None)
df.head(1)


,case_id,title,number,type,instrument,dg,language,cartel,legal_basis_code,legal_basis_label,sector_code,sector_label,companies,initiation_date,last_decision_date,caseInstrument,caseNumber,caseTitle,caseSectorsCode,caseSectorLabel,caseCompanies,caseLegalBasisCode,caseLegalBasisLabel,caseLastDecisionDate,caseInitiationDate,decisions,caseAttachments,num_case_attachments,num_decisions,num_decision_attachments,decision_type_labels,decision_adoption_dates,decision_publication_dates,press_release_publication_dates
0,M.2027,DEUTSCHE BANK / SAP / JV,2027,None,Merger,None,None,None,,Art. 105,NaceSectorsG_46,"G.46 - Wholesale trade, except of motor vehicl...",DEUTSCHE BANK; SAP; JV,2000-06-09,2000-07-13,Merger,2027,DEUTSCHE BANK / SAP / JV,NaceSectorsG_46,"G.46 - Wholesale trade, except of motor vehicl...","[DEUTSCHE BANK, SAP, JV]",,[Art. 105],2000-07-13,2000-06-09,"[{'decisionDate': '2000-07-13', 'decisionLabel...",[],0,2,0,[Art. 6(1)(b)],[2000-07-13],[],[]


In [ ]:
df

,case_id,title,number,type,instrument,dg,language,cartel,legal_basis_code,legal_basis_label,sector_code,sector_label,companies,initiation_date,last_decision_date,metadata,caseAttachments,decisions,_sourceFile,num_case_attachments,num_decisions,num_decision_attachments,decision_type_labels,decision_adoption_dates,decision_publication_dates,press_release_publication_dates
0,AT.35803,IPEX Consortium,35803,AtStandardATCCase,Antitrust & Cartels,Competition DG,en,Antitrust,AtLegalBase3,Art. 101 TFEU,NaceSectorsH_50_2_0,H.50.20 - Sea and coastal freight water transport,Andrew Weir Shipping; DSR-Senator Lines; Compa...,1995-10-20,None,"{'caseLastDecisionDate': [], 'caseExternalLink...","[{'metadata': {'attachmentCategory': ['{""code""...",[],AT,1,0,0,[],[],[],[]
1,AT.34950,ECO EMBALLAGES,34950,AtStandardATCCase,Antitrust & Cartels,Competition DG,en,Antitrust,AtLegalBase3,Art. 101 TFEU,NaceSectorsE_38_3,E.38.3 - Materials recovery,,1993-12-17,2001-06-15,"{'caseLastDecisionDate': ['2001-06-15'], 'case...",[],"[{'metadata': {'language': ['en'], 'metadataRe...",AT,0,1,0,[Old milestones - Negative clearance decision],[2001-06-15],[2001-06-13],[2001-06-13]
2,AT.39172,Electricity sector inquiry,39172,AtSectorInquiryCase,Antitrust & Cartels,Competition DG,en,Antitrust,None,None,NaceSectorsD_35_1,"D.35.1 - Electric power generation, transmissi...",,2005-03-04,2007-01-10,"{'caseLastDecisionDate': ['2007-01-10'], 'case...",[],"[{'metadata': {'language': ['en'], 'metadataRe...",AT,0,1,0,[Sector Inquiry - Final Report],[2007-01-10],[2007-01-10],[2007-01-10]
3,AT.39294,Microsoft (ECIS complaint),39294,AtStandardATCCase,Antitrust & Cartels,Competition DG,en,Antitrust,AtLegalBase101102,Art. 101 TFEU + Art. 102 TFEU,NaceSectorsC_26_2,C.26.2 - Manufacture of computers and peripher...,Microsoft Corporation; European Committee for ...,2006-02-22,None,"{'caseLastDecisionDate': [], 'caseExternalLink...","[{'metadata': {'attachmentCategory': ['{""code""...",[],AT,2,0,0,[],[],[],[]
4,AT.39173,Gas sector inquiry,39173,AtSectorInquiryCase,Antitrust & Cartels,Competition DG,en,Antitrust,None,None,NaceSectorsD_35_2,D.35.2 - Manufacture of gas; distribution of g...,,2005-03-04,2007-01-10,"{'caseLastDecisionDate': ['2007-01-10'], 'case...",[],"[{'metadata': {'language': ['en'], 'metadataRe...",AT,0,1,0,[Sector Inquiry - Final Report],[2007-01-10],[2007-01-10],[2007-01-10]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71808,SA.122902,Atbalsta shēmā “Līdzfinansējums dalībai kopste...,122902,GlobalExemptionAGRICase,State Aid,Competition DG,en,None,None,None,None,None,,2026-04-02,None,"{'caseLastDecisionDate': [], 'caseSecondaryLaw...",[{'metadata': {'attachmentCategory': ['Documen...,[],SA,24,0,0,[],[],[],[]
71809,SA.122903,"STEP technologijų kūrimas, skiriant alternatyv...",122903,GlobalExemptionCOMPCase,State Aid,Competition DG,en,None,None,None,None,None,,2026-04-03,None,"{'caseLastDecisionDate': [], 'caseSecondaryLaw...",[{'metadata': {'attachmentCategory': ['Documen...,[],SA,24,0,0,[],[],[],[]
71810,SA.122905,2021–2030 m. energetikos plėtros programos paž...,122905,GlobalExemptionCOMPCase,State Aid,Competition DG,en,None,None,None,None,None,,2026-04-03,None,"{'caseLastDecisionDate': [], 'caseSecondaryLaw...",[{'metadata': {'attachmentCategory': ['Documen...,[],SA,19,0,0,[],[],[],[]
71811,SA.122906,Dotacja na zachowanie dziedzictwa kulturowego ...,122906,GlobalExemptionCOMPCase,State Aid,Competition DG,en,None,None,None,None,None,,2026-04-03,None,"{'caseLastDecisionDate': [], 'caseSecondaryLaw...",[{'metadata': {'attachmentCategory': ['Documen...,[],SA,24,0,0,[],[],[],[]


In [ ]:
df["metadata"][0]

{'caseLastDecisionDate': [],
 'caseExternalLinks': ['{"items":[{}]}'],
 'caseLinks': [],
 'caseInstrument': ['Antitrust & Cartels'],
 'language': ['en'],
 'metadataReference': ['AT.35803'],
 'caseNumberPart': ['35803'],
 'caseType': ['AtStandardATCCase'],
 'caseNumber': ['AT.35803'],
 'casePressReleases': ['{"items":[{}]}'],
 'caseTimelineEvents': ['{"items":[{}]}'],
 'caseInitiationDate': ['1995-10-20'],
 'caseCourtCases': [],
 'caseOfficialJournalPublications': ['{"items":[{}]}'],
 'caseCartel': ['Antitrust'],
 'caseLegalBasis': ['{"code":"AtLegalBase3","label":"Art. 101 TFEU"}'],
 'caseCompanies': ['Andrew Weir Shipping',
  'DSR-Senator Lines',
  "Compagnie Maritime d'affretement"],
 'caseTitle': ['IPEX Consortium'],
 'caseDg': ['Competition DG'],
 'caseOfficialJournalPublicationsPublishedDates': [],
 'caseTimelineEventsDates': [],
 'casePressReleasesPublicationDates': [],
 'caseSectors': ['{"code":"NaceSectorsH_50_2_0","label":"H.50.20 - Sea and coastal freight water transport"}'],

In [ ]:
df["legal_basis_label"].unique()

array(['Art. 101 TFEU', None, 'Art. 101 TFEU + Art. 102 TFEU',
       'Art. 101 TFEU + Art. 53 EEA', 'Art 65 ECSC Treaty',
       'Art. 102 TFEU', 'Art. 102 + Art. 37', 'Art. 106 + Art. 102',
       'Art. 102 TFEU + Art. 54 EEA',
       'Art. 101 TFEU + Art. 102 TFEU + Art. 7 Reg.2003/1228',
       'Art. 102 TFEU + Art. 106 + Art. 4 TFEU', 'Art. 106 + Art. 101',
       'Art 258 TFEU (Ex 226 EC)', 'Art 106 TFEU (Ex 86 EC)',
       'Art 105 TFEU (Ex 85 EC)',
       'Art. 101 + Art. 102 + Art. 53 + Art. 54',
       'Art 23(1)e Regulation 2003/1', 'Art. 106 + Art. 101 + Art. 102',
       'Art. 101 TFEU + Art. 102 TFEU + Art. 106 + Art. 4 TFEU',
       'Art. 101 TFEU + Art. 102 TFEU + Art. 105 TFEU',
       'Art. 101 TFEU + Art. 105 TFEU', 'DMA – Regulation 2022/1925',
       '3(3) DMA - Quantitative designation (Notification)',
       '3(8) DMA – Qualitative designation',
       '19 DMA - Market investigation into new services and new practices'],
      dtype=object)

In [ ]:
def keep_art_legal_bases(value):
    if pd.isna(value):
        return value

    parts = [part.strip() for part in str(value).split("+")]
    art_parts = [part for part in parts if part.startswith("Art")]

    return " + ".join(art_parts) if art_parts else None


df["legal_basis_label"] = df["legal_basis_label"].apply(keep_art_legal_bases)

# Drop rows where no legal basis started with "Art"
df = df[df["legal_basis_label"].notna()].copy()

df["legal_basis_label"].unique()

In [ ]:
sorted({label for labels in df["decision_type_labels"] for label in labels})


['2/3 rule',
 'Aborted / withdrawn',
 'Aborted / withdrawn (N/2)',
 'Amending Decision',
 'Art. 10(3)',
 'Art. 10(3) FSR Decision to open the in-depth investigation',
 'Art. 11(3) FSR Decision with Commitments',
 'Art. 14',
 'Art. 18 (Statement of Objections)',
 'Art. 19 (Advisory Committee)',
 'Art. 21(3)',
 'Art. 21(4)',
 'Art. 21(4) Preliminary Assessment',
 'Art. 22 Full referral',
 'Art. 22 Refusal of referral',
 'Art. 22(4)',
 'Art. 24(5) FSR Stop-the-clock decision',
 'Art. 4(4) - Full referral',
 'Art. 4(4) - No referral',
 'Art. 4(4) - Partial referral',
 'Art. 4(5) - Referral',
 'Art. 5(2) - Incompleteness',
 'Art. 5(3) - Emergence of new facts during procedure',
 'Art. 6(1)(a)',
 'Art. 6(1)(b)',
 'Art. 6(1)(b) with conditions & obligations',
 'Art. 6(1)(c)',
 'Art. 7(3)',
 'Art. 7(3) refusal',
 'Art. 8(1)',
 'Art. 8(2)',
 'Art. 8(2) with conditions & obligations',
 'Art. 8(3)',
 'Art. 8(4)',
 'Art. 8(4)(b)',
 'Art. 8(4)a',
 'Art. 8(5)(a)',
 'Art. 8(5)(c)',
 'Art. 9(3) full r

In [ ]:

arr =  ['Old milestones - Negative clearance decision',
       'Sector Inquiry - Final Report', 'Rejection of Complaint Decision',
       'Closure of Proceedings', 'Commitment Decision',
       'Initiation of Proceedings', 'Cooperation Decision',
       'Prohibition Decision', 'State Measure Decision',
       'Old milestones - Exemption with condition decision',
       'Settlement Decision', 'Amending Decision',
       'Decision imposing fines', 'Interim Measures Decision',
       'Informal Guidance Letter',
       'Decision to open proceedings based on Art.8(2)',
       'Decision under waiver clause', 'Decision based on Art.8',
       'Preliminary findings based on Art.8(5)', 'Designation decision',
       'Decision accepting rebuttal',
       'Decision to open a market investigation based on Art.3(8) and Art.17(1)',
       'Decision to open proceedings based on Art.20 and Art.29',
       'Review of designation decision based on Art.4',
       'Decision to open a market investigation into new services and new practices based on Art.19',
       'Decision to open a market investigation based on Art.3(5) and Art.17(3)',
       'Non-compliance decision based on Art.29', 'Closure decision',
       'Decision to extend time limits based on Art.7(6)',
       'The provisional deadline and the suspension period under FSR expired on',
       'Withdrawal during preliminary review',
       'Art. 11(3) FSR Decision with Commitments',
       'Art. 10(3) FSR Decision to open the in-depth investigation',
       'Art. 24(5) FSR Stop-the-clock decision', 'Art. 6(1)(b)',
       'Art. 6(1)(b) with conditions & obligations', 'Art. 6(1)(a)',
       'Art. 6(1)(c)', 'Art. 18 (Statement of Objections)',
       'Art. 8(2) with conditions & obligations',
       'Art. 9(3) partial referral', 'Withdrawn', 'Art. 21(3)',
       'Art. 8(4)', 'Art. 8(3)', 'Art. 22(4)', 'Art. 8(2)', 'Art. 14',
       'Art. 9(3) full referral', 'Art. 19 (Advisory Committee)',
       'Aborted / withdrawn', 'Art. 7(3)',
       'Modification of Art. 6(1)(b) with conditions & obligations',
       'Aborted / withdrawn (N/2)',
       'Modification of Art. 8(2) with conditions & obligations',
       'Art. 8(1)', 'Art. 9(3) refusal of referral',
       'Rejection of a request to act ', 'Art. 21(4)',
       'Art. 22 Full referral', 'Art. 4(4) - Full referral',
       'Art. 5(2) - Incompleteness', 'Art. 4(4) - Partial referral',
       'Decision under remedy review clause', 'Art. 10(3)',
       'Art. 22 Refusal of referral',
       'Art. 5(3) - Emergence of new facts during procedure',
       'Decision on the implementation of remedies', '2/3 rule',
       'No substance', 'Purchaser approval', 'Art. 7(3) refusal',
       'Closure of proceedings', 'Rejection of a request to act',
       'Art. 8(4)(b)', 'Withdrawal of Art. 6(1)(c) decision',
       'Withdrawal of Art. 8(3) decision', 'Art. 4(5) - Referral',
       'Art. 4(4) - No referral', 'Withdrawal of Art. 8(5)(c) decision',
       'Art. 8(5)(c)', 'Withdrawal of Art. 14 decision',
       'Art. 21(4) Preliminary Assessment', 'Implementing decision',
       'Art. 8(4)a', 'Withdrawal of Art. 8(4)(a) decision',
       'Art. 8(5)(a)', 'Withdrawal of Art. 8(5)(a) decision',
       'Decision not to raise objections', 'Positive decision',
       'Corrigendum',
       'Decision finding that the measures do not constitute aid',
       'Withdrawal of notification (after formal investigation procedure)',
       'Decision to initiate the formal investigation procedure',
       'Negative decision on notified aid not put into effect',
       'Negative decision without recovery',
       'Decision to extend proceedings', 'Information injunction',
       'Negative decision with recovery',
       'Referral to Court of Justice (non-compliance with decisions)',
       'Decision finding that the measures do not constitute aid (after formal investigation procedure)',
       'Proposal for appropriate measures', 'Conditional decision',
       'Expected',
       'EC Treaty - referral to the Court of Justice (non-compliance with Court judgment)',
       'Revocation of a decision', 'Other',
       'Referral to the Court of Justice (non-implementation of Commission decision) (ex Article 88(2) EC)',
       'Withdrawal of notification (before formal investigation procedure)',
       'Referral to the Court of Justice (non-compliance with Court judgment) – ex article 228(2) EC Treaty',
       'Referral to Court of Justice (non-compliance with injunction)',
       'Suspension injunction',
       'No aid decision (after formal investigation procedure)',
       'Corrigendum or Correcting decision',
       'Decision to close formal investigation procedure without object',
       'GBER – approval of the evaluation plan']

import re
unique_reasons = set()

for item in arr:
    if not item:
        continue
    # Split on '+', clean spaces, normalize formatting
    parts = [re.sub(r'\s+', ' ', p.strip().replace('Art ', 'Art. ')) for p in item.split('+')]
    unique_reasons.update(parts)

# Sort for readability
unique_reasons = sorted(unique_reasons)

for u in unique_reasons:
    print(u)


2/3 rule
Aborted / withdrawn
Aborted / withdrawn (N/2)
Amending Decision
Art. 10(3)
Art. 10(3) FSR Decision to open the in-depth investigation
Art. 11(3) FSR Decision with Commitments
Art. 14
Art. 18 (Statement of Objections)
Art. 19 (Advisory Committee)
Art. 21(3)
Art. 21(4)
Art. 21(4) Preliminary Assessment
Art. 22 Full referral
Art. 22 Refusal of referral
Art. 22(4)
Art. 24(5) FSR Stop-the-clock decision
Art. 4(4) - Full referral
Art. 4(4) - No referral
Art. 4(4) - Partial referral
Art. 4(5) - Referral
Art. 5(2) - Incompleteness
Art. 5(3) - Emergence of new facts during procedure
Art. 6(1)(a)
Art. 6(1)(b)
Art. 6(1)(b) with conditions & obligations
Art. 6(1)(c)
Art. 7(3)
Art. 7(3) refusal
Art. 8(1)
Art. 8(2)
Art. 8(2) with conditions & obligations
Art. 8(3)
Art. 8(4)
Art. 8(4)(b)
Art. 8(4)a
Art. 8(5)(a)
Art. 8(5)(c)
Art. 9(3) full referral
Art. 9(3) partial referral
Art. 9(3) refusal of referral
Closure decision
Closure of Proceedings
Closure of proceedings
Commitment Decision
Condit

In [ ]:
df

In [ ]:
df['legal_basis_label'].isnull().sum()


In [ ]:
df["legal_basis_code"] == None

In [ ]:
# Older per-file/per-table loading has been replaced by the single DataFrame `df`.
# `df` keeps nested JSON columns and includes normalized case-level columns for analysis.
df.head()


# Follow up analysis

In [ ]:
df['legal_basis_label'] = df['legal_basis_label'].apply(
    lambda x: ";".join(x.split("+")) if isinstance(x, str) else x
)
with pd.option_context('display.max_rows', None):
    print(df["legal_basis_label"])

In [ ]:
print("Number of Laws:", len(df["legal_basis_label"].unique())-1)

In [ ]:
print("Laws:", df["legal_basis_label"].unique())

In [ ]:
df.head(1)

In [ ]:
# how many missing decisions
missing_cases = (df["num_decisions"] == 0).sum()
print("Number of cases that has no decisions:", missing_cases)


In [ ]:
len(df)

In [ ]:
# how many cases with more than one decision
num_cases_with_multiple_decisions = (df["num_decisions"] > 1).sum()
print("Number of cases with more than one decision:", num_cases_with_multiple_decisions)


In [ ]:
df[df["case_id"]  == "AT.39850"]

In [ ]:
df.loc[df["case_id"] == "AT.39850", ["case_id", "title", "num_decisions", "decisions"]]


In [ ]:
multi_decision_cases

In [ ]:
# when are cases initiated
import pandas as pd
import matplotlib.pyplot as plt

df['initiation_date'] = pd.to_datetime(df['initiation_date'], utc=True)

df['year'] = df['initiation_date'].dt.year
df['year'].value_counts().sort_index().plot(kind='bar')
plt.title('Number of Cases per Year')
plt.xlabel('Year')
plt.ylabel('Cases')
plt.show()


In [ ]:
df[["case_id", "title", "num_decisions", "decision_type_labels", "decision_publication_dates"]].head()


In [ ]:
# Decision dates are stored in df["decision_publication_dates"].
# Use explode only for this plot/analysis when a per-decision view is needed.
decision_years = pd.to_datetime(
    df["decision_publication_dates"].explode(), utc=True, errors="coerce"
).dt.year.dropna()
decision_years.value_counts().sort_index().plot(kind="bar")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert dates
df["initiation_date"] = pd.to_datetime(df["initiation_date"], utc=True, errors="coerce")

# Extract years
df["year"] = df["initiation_date"].dt.year
decision_years = pd.to_datetime(
    df["decision_publication_dates"].explode(), utc=True, errors="coerce"
).dt.year.dropna()

# Count per year
cases_per_year = df["year"].value_counts().sort_index()
decisions_per_year = decision_years.value_counts().sort_index()

# Combine into one DataFrame for aligned plotting
combined = pd.DataFrame({
    "Cases": cases_per_year,
    "Decisions": decisions_per_year,
}).fillna(0)

combined.plot(kind="line", marker="o", figsize=(10, 6))
plt.title("Cases vs Decisions per Year (Trend)")
plt.xlabel("Year")
plt.ylabel("Count")
plt.grid(True)
plt.xticks(rotation=45)
plt.legend(title="Type")
plt.tight_layout()
plt.show()


In [ ]:
today = pd.Timestamp.now(tz='UTC')
df['case_age_days'] = (today - df['initiation_date']).dt.days
df['case_age_years'] = df['case_age_days'] // 365

In [ ]:
plt.figure(figsize=(10, 6))
df['case_age_years'].plot(kind='hist', bins=30, edgecolor='black')
plt.title('Distribution of Case Ages (Years)')
plt.xlabel('Age in Years')
plt.ylabel('Number of Cases')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
df["initiation_date"] = pd.to_datetime(df["initiation_date"], utc=True, errors="coerce")

decision_timing_df = df[["case_id", "initiation_date", "decision_publication_dates"]].explode("decision_publication_dates")
decision_timing_df["publication_date"] = pd.to_datetime(
    decision_timing_df["decision_publication_dates"], utc=True, errors="coerce"
)
decision_timing_df = decision_timing_df.dropna(subset=["initiation_date", "publication_date"])
decision_timing_df["decision_delay_days"] = (
    decision_timing_df["publication_date"] - decision_timing_df["initiation_date"]
).dt.days


In [ ]:
# Days
min_days = merged_df['decision_delay_days'].min()
avg_days = merged_df['decision_delay_days'].mean()
max_days = merged_df['decision_delay_days'].max()

# Convert to years
min_years = min_days / 365
avg_years = avg_days / 365
max_years = max_days / 365

# Print both
print(f"Minimum delay: {min_days} days ({min_years:.2f} years)")
print(f"Average delay: {avg_days:.2f} days ({avg_years:.2f} years)")
print(f"Maximum delay: {max_days} days ({max_years:.2f} years)")


# Number of cases that has no Companies, Sectors, or Legal Basis

In [ ]:
len(df)

In [ ]:
#Number of cases that has no Companies
missing_companies = df[
    df['companies'].isna() | (df['companies'].str.strip() == '')
]
print("Number of cases that has no Companies:", len(missing_companies))

In [ ]:
#Number of cases that has no sector
missing_laws = df[
    df['sector_label'].isna() | (df['sector_label'].str.strip() == '')
]
print("Number of cases that has no Sector:", len(missing_laws))

In [ ]:
#Number of cases that has no Legal basis
missing_laws = df[
    df['legal_basis_label'].isna() | (df['legal_basis_label'].str.strip() == '')
]
print("Number of cases that has no Legal basis:", len(missing_laws))

# Company Names

In [ ]:
# STEP 1: Clean and split company names
df['companies_list'] = df['companies'].str.split(';')

# Remove whitespace and empty entries
df['companies_list'] = df['companies_list'].apply(
    lambda x: [c.strip() for c in x if c.strip()] if isinstance(x, list) else []
)

# STEP 2: Explode into one company per row
exploded_df = df.explode('companies_list').rename(columns={'companies_list': 'company'})

# STEP 3: Count unique companies
unique_company_count = exploded_df['company'].nunique()
print(f"Number of unique companies: {unique_company_count}")

# STEP 4: Count occurrences of each company
company_counts = exploded_df['company'].value_counts()

In [ ]:
with pd.option_context('display.max_rows', None):
    print()

In [ ]:
# # STEP 5: Plot company frequencies (can get large if too many)
# plt.figure(figsize=(20, 10))
# company_counts.plot(kind='bar')
# plt.title('Company Involvement Frequency')
# plt.xlabel('Company')
# plt.ylabel('Number of Cases')
# plt.xticks(rotation=90)
# plt.tight_layout()
# plt.show()


In [ ]:
print("Number of unique sectors:", df["sector_label"].nunique())

In [ ]:
print("Number of unique Legal Basis:", df["legal_basis_label"].nunique())

# cases without decision

In [ ]:
# Cases with no decisions stay in the single DataFrame.
cases_without_decisions = df[df["num_decisions"] == 0]
print(f"Number of cases without decisions: {len(cases_without_decisions)}")


In [ ]:
cases_without_decisions.head()


In [ ]:
cases_without_decisions['year'] = cases_without_decisions['initiation_date'].dt.year
cases_without_decisions['year'].value_counts().sort_index().plot(kind='bar', figsize=(8, 4), color='orange')
plt.title('Cases Without Decisions by Year')
plt.xlabel('Year')
plt.ylabel('Number of Undecided Cases')
plt.tight_layout()
plt.show()


# Number of decisions per case

In [ ]:
# `num_decisions` is computed while loading the single DataFrame.
df["num_decisions"].value_counts().sort_index()


In [ ]:
plt.figure(figsize=(8, 5))
df['num_decisions'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribution of Number of Decisions per Case')
plt.xlabel('Number of Decisions')
plt.ylabel('Number of Cases')
plt.grid(axis='y')
plt.tight_layout()
plt.show()


# does press releases corelate with decisions

In [ ]:
# Decision attachments are kept nested inside df["decisions"].
df[["case_id", "num_decision_attachments", "decisions"]].head()


In [ ]:
# Temporary decision-level view, derived from df only when needed.
decision_view = decision_records_from_df(df)
decision_view.head()


In [ ]:
decision_view = decision_records_from_df(df)
missing_press_releases = decision_view["press_release_publication_date"].isna().sum()
print("Decisions without press release:", missing_press_releases)


In [ ]:
decision_view = decision_records_from_df(df)
decision_view["adoption_date"] = pd.to_datetime(decision_view["adoption_date"], utc=True, errors="coerce")
decision_view["press_release_publication_date"] = pd.to_datetime(
    decision_view["press_release_publication_date"], utc=True, errors="coerce"
)


In [ ]:
decision_view["press_delay_days"] = (
    decision_view["press_release_publication_date"] - decision_view["adoption_date"]
).dt.days


In [ ]:
print("\nNumber of decisions with same-day press release:", (decision_view["press_delay_days"] == 0).sum())
print("Number of decisions with delayed press release:", (decision_view["press_delay_days"] > 0).sum())
print("Number of decisions with press release before adoption:", (decision_view["press_delay_days"] < 0).sum())


In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(decision_view["adoption_date"], decision_view["press_delay_days"], alpha=0.7, color="green")
plt.title("Press Release Delay Over Time")
plt.xlabel("Adoption Date")
plt.ylabel("Delay in Days")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Raw JSON stays available inside the nested DataFrame columns.
df.loc[df["case_id"].str.startswith("AT."), ["case_id", "metadata", "caseAttachments", "decisions"]].head()


In [ ]:
# One-row-per-case DataFrame, not one file or one table per source.
df.head()
